In [1]:
# Importing the necessary Libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Loading the Competitive Matches and National Teams data

base_dir = Path.cwd()
while base_dir.name != "World Cup Predictor":
    base_dir = base_dir.parent

competitive_matches_path = base_dir/"data"/"processed"/"clean_competitive_matches.csv"
competitive_matches_df = pd.read_csv(competitive_matches_path)
national_teams_path = base_dir/"data"/"raw"/"national_teams.csv"
national_teams_df = pd.read_csv(national_teams_path)
elo_ratings_path = base_dir/"data"/"raw"/"elo_ratings.csv"
elo_ratings_df = pd.read_csv(elo_ratings_path)

In [3]:
# Converting date column to datetime
competitive_matches_df["date"] = pd.to_datetime(competitive_matches_df["date"])
name_mapping = {'Czechoslovakia': 'Czech Republic'}
competitive_matches_df['home_team'] = competitive_matches_df['home_team'].replace(name_mapping)
competitive_matches_df['away_team'] = competitive_matches_df['away_team'].replace(name_mapping)

#  Calculate a decay weight for each match: matches closer to today get weight ~1.0, older matches get exponentially lower weight. Formula: e^(-λ * years_ago) where λ is the decay rate (use 0.1 as a starting value)
today_date = pd.Timestamp.now()
time_delta = (today_date - competitive_matches_df["date"]) / pd.Timedelta(days=365.25) # (Converting to years)
decay_weight = np.exp(-0.175*time_delta)

# Adding decay_weight as a column to competitive_matches_df before splitting into home/away
competitive_matches_df['weight'] = decay_weight


# Segregating into two dataframes - home and away
home_df = competitive_matches_df[['home_team', 'home_score', 'away_score', 'weight']]
away_df = competitive_matches_df[['away_team', 'home_score', 'away_score', 'weight']]

In [4]:
# Renaming the column names

home_df = home_df.rename(columns={
    'home_team':'team',
    'home_score':'goals_for',
    'away_score':'goals_against'
})
away_df = away_df.rename(columns={
    'away_team':'team',
    'away_score':'goals_for',
    'home_score':'goals_against'
})

In [5]:
# Concatenating both dataframes into one

combined_df = pd.concat([home_df, away_df], ignore_index=True)

In [6]:
# Calculating Team Strength by taking the average of goals scored and conceded by all teams

def weighted_mean(group):
    return pd.Series({
        'goals_for': (group['goals_for'] * group['weight']).sum() / group['weight'].sum(),
        'goals_against': (group['goals_against'] * group['weight']).sum() / group['weight'].sum()
    })

team_strength_df = pd.concat([home_df, away_df], ignore_index = True). groupby('team').apply(weighted_mean) 

In [7]:
# Check the team strengths

print(team_strength_df.head())

             goals_for  goals_against
team                                 
Abkhazia      1.799190       0.735458
Afghanistan   0.928274       1.718778
Albania       1.049334       1.062597
Alderney      0.730836       3.707662
Algeria       2.088130       0.812804


In [8]:
# Using National Teams data to factor in the elo ratings, total market value and fifa rankings.
national_teams = national_teams_df[['name', 'total_market_value', 'fifa_ranking']]

# Using log scaling and min-max normalizing to reward the stronger squads.
national_teams['log_market_value'] = np.log(national_teams['total_market_value'])

squad_value = national_teams['log_market_value']
log_min = squad_value.min()
log_max = squad_value.max()
national_teams['squad_value_weight'] = 0.2 + 1.8 * (squad_value - log_min) / (log_max - log_min) # Apply min-max normalization on the log_market_value column scaled to 0.2-1.0

rank = national_teams['fifa_ranking']
rank_max = rank.max()
national_teams['fifa_ranking_weight'] = 0.2 + 1.8 * (rank_max - rank) / (rank_max - 1)

elo = elo_ratings_df['elo_rating']
elo_min = elo.min()
elo_max = elo.max()
elo_ratings_df['elo_rating_weight'] = 0.2 + 1.8 * (elo - elo_min) / (elo_max - elo_min)

In [9]:
# Merging the Elo Ratings DataFrame into National teams DataFrame
national_teams = national_teams.reset_index()
national_teams = pd.merge(national_teams, elo_ratings_df[['name','elo_rating_weight']], left_on='name', right_on='name', how='left')
national_teams['elo_rating_weight'] = national_teams['elo_rating_weight'].fillna(0.2)

# Calculating the National teams weight based on all the 3 individual weights
national_teams['weight'] = 0.3 * national_teams['squad_value_weight'] + 0.3 * national_teams['fifa_ranking_weight'] + 0.4 * national_teams['elo_rating_weight']

In [10]:
 print(national_teams[national_teams['name'].isin(['France', 'England'])][['name', 'elo_rating_weight']])

      name  elo_rating_weight
0   France           1.914237
3  England           1.851957


In [11]:
# Merging the National teams DataFrame into Team Strength DataFrame
team_strength_df = team_strength_df.reset_index()
team_strength_df = pd.merge(team_strength_df, national_teams[['name','weight']], left_on='team', right_on='name', how='left')

In [12]:
team_strength_df['weight'] = team_strength_df['weight'].fillna(0.2)

In [13]:
print(national_teams[national_teams['name'].isin(['France', 'England', 'Czechia'])]['weight'])
print(national_teams[national_teams['name'].isin(['France', 'England', 'Czechia'])][['name', 'squad_value_weight', 'fifa_ranking_weight', 'elo_rating_weight']])
national_teams_df[national_teams_df['total_market_value'].isna()]['name']

0    1.961321
3    1.933068
Name: weight, dtype: float64
      name  squad_value_weight  fifa_ranking_weight  elo_rating_weight
0   France             1.98542             2.000000           1.914237
3  England             2.00000             1.974286           1.851957


Series([], Name: name, dtype: str)

In [14]:
# Apply the weight to goals_for and goals_against
'''
Explanation: 
- Strong team (weight=2.0): goals_against / 2.0 → half as easy to score against ✓
- Weak team (weight=0.2): goals_against / 0.2 → 5× easier to score against ✓
'''

team_strength_df['goals_for'] = team_strength_df['goals_for'] * team_strength_df['weight']
team_strength_df['goals_against'] = team_strength_df['goals_against'] / team_strength_df['weight']
team_strength_df = team_strength_df[['team','goals_for','goals_against']].set_index('team')

In [15]:
# print(team_strength_df.index[:5])
# print(team_strength_df[250:300])
print(team_strength_df.loc[['Brazil', 'France', 'England', 'Germany', 'Ecuador', 'Panama', 'Iran']])

         goals_for  goals_against
team                             
Brazil    3.281001       0.376771
France    4.028935       0.424358
England   4.465098       0.311803
Germany   4.615090       0.544560
Ecuador   1.942145       0.506406
Panama    2.623519       0.703371
Iran      3.321359       0.434672


In [16]:
# Save the Team Strength dataframe to the processed data folder

team_strength_path = base_dir/"data"/"processed"/"team_strength.csv"
team_strength_df.to_csv(team_strength_path, index = True)